# Pixels to Predictions — v2 Final (HPC A100 Optimized)

## Recipe
- 0.859 team base: r=8, all 7 targets, flat LR=2e-4, 3 epochs, train+val combined
- Our additions: self-captioning, DoRA, subject/grade metadata, solution field, fixed inference, 5-model ensemble

## Hardware: A100 40GB, 12 cores
- Training batch=4, grad_accum=2 (effective=8)
- Inference batch=16
- num_workers=4
- Estimated: ~3-4 hours total

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1: Setup Paths
# ══════════════════════════════════════════════════════════════════════════════
# For HPC: set these to your actual paths
# For Colab: uncomment the drive mount block

import os
from pathlib import Path

# ── OPTION A: Google Colab ──
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = Path("/content/drive/MyDrive/kaggle_final_competition")

# ── OPTION B: HPC (edit this path) ──
PROJECT_ROOT = Path("/scratch/au2414/kaggle_final_competition")  # <-- CHANGE THIS

DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_v2"
SUBMISSION_DIR = PROJECT_ROOT / "submissions_v2"
CAPTION_DIR = PROJECT_ROOT / "captions"

for d in [CHECKPOINT_DIR, SUBMISSION_DIR, CAPTION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f"Data dir not found: {DATA_DIR}"
print("Data:", sorted(os.listdir(DATA_DIR)))
print("Checkpoints:", sorted(os.listdir(CHECKPOINT_DIR)))

Data: ['images', 'sample_submission.csv', 'test.csv', 'train.csv', 'val.csv']
Checkpoints: ['.ipynb_checkpoints', 'A_cap_meta_dora_r8a16_s42_ep1_b260', 'A_cap_meta_dora_r8a16_s42_ep1_b520', 'A_cap_meta_dora_r8a16_s42_ep1_b780', 'A_cap_meta_dora_r8a16_s42_ep1_final', 'A_cap_meta_dora_r8a16_s42_ep2_b260', 'A_cap_meta_dora_r8a16_s42_ep2_b520', 'A_cap_meta_dora_r8a16_s42_ep2_b780', 'A_cap_meta_dora_r8a16_s42_ep2_final', 'A_cap_meta_dora_r8a16_s42_ep3_b260', 'A_cap_meta_dora_r8a16_s42_ep3_b520', 'A_cap_meta_dora_r8a16_s42_ep3_b780', 'A_cap_meta_dora_r8a16_s42_ep3_final', 'A_cap_meta_dora_r8a16_s42_final', 'A_cap_meta_dora_r8a16_s42_log.json', 'A_cap_meta_dora_r8a16_s42_results.json', 'B_cap_meta_dora_r8a16_s123_ep1_b260', 'B_cap_meta_dora_r8a16_s123_ep1_b520', 'B_cap_meta_dora_r8a16_s123_ep1_b780', 'B_cap_meta_dora_r8a16_s123_ep1_final', 'B_cap_meta_dora_r8a16_s123_ep2_b260', 'B_cap_meta_dora_r8a16_s123_ep2_b520', 'B_cap_meta_dora_r8a16_s123_ep2_b780', 'B_cap_meta_dora_r8a16_s123_ep2_final'

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2: Install
# ══════════════════════════════════════════════════════════════════════════════
!pip install -q transformers==4.57.6 peft==0.17.1 bitsandbytes accelerate datasets pillow

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3: Imports & GPU Config
# ══════════════════════════════════════════════════════════════════════════════
import json, random, time, gc, copy
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"
MAX_CONTEXT_CHARS = 400

# ── A100-optimized settings ──
TRAIN_BATCH = 4
GRAD_ACCUM = 2        # effective batch = 4 * 2 = 8
INFER_BATCH = 16
NUM_WORKERS = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} | VRAM: {vram:.1f} GB")
    USE_BF16 = torch.cuda.is_bf16_supported()
    COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    print(f"Compute dtype: {COMPUTE_DTYPE}")
    print(f"Training: batch={TRAIN_BATCH} x accum={GRAD_ACCUM} = effective {TRAIN_BATCH*GRAD_ACCUM}")
    print(f"Inference: batch={INFER_BATCH}")
    print(f"Workers: {NUM_WORKERS}")
else:
    COMPUTE_DTYPE = torch.float32
    print("WARNING: No GPU!")

/home/au2414/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4: Load Data (train+val COMBINED)
# ══════════════════════════════════════════════════════════════════════════════
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

# CRITICAL: combine train + val
combined_df = pd.concat([train_df, val_df], ignore_index=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Combined: {len(combined_df)} | Test: {len(test_df)}")
print(f"Solutions available: {combined_df['solution'].notna().sum()}/{len(combined_df)}")
print(f"\nTraining on train+val combined. Blind submit — no validation.")

Train: 3109 | Val: 1048 | Combined: 4157 | Test: 1008
Solutions available: 3456/4157

Training on train+val combined. Blind submit — no validation.


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5: Load or Generate Captions
# ══════════════════════════════════════════════════════════════════════════════
from transformers import AutoProcessor, AutoModelForVision2Seq

caption_file = CAPTION_DIR / "all_captions.json"

if caption_file.exists():
    with open(caption_file) as f:
        all_captions = json.load(f)
    print(f"Loaded {len(all_captions)} cached captions.")
else:
    print("No cached captions. Generating...")

    cap_processor = AutoProcessor.from_pretrained(MODEL_ID)
    if cap_processor.tokenizer.pad_token is None:
        cap_processor.tokenizer.pad_token = cap_processor.tokenizer.eos_token

    cap_model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID, torch_dtype=COMPUTE_DTYPE,
        device_map="auto", low_cpu_mem_usage=True,
    )
    cap_model.eval()

    cap_prompt = "<image>\nDescribe this image in detail. Focus on any diagrams, charts, maps, labels, numbers, arrows, or scientific content visible."
    all_captions = {}

    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        print(f"\nCaptioning {split_name} ({len(split_df)} images)...")
        pbar = tqdm(range(0, len(split_df), 8), desc=split_name)
        for start in pbar:
            end = min(start + 8, len(split_df))
            chunk = split_df.iloc[start:end]

            images = [Image.open(DATA_DIR / row["image_path"]).convert("RGB")
                      for _, row in chunk.iterrows()]
            prompts = [cap_prompt] * len(images)

            inputs = cap_processor(text=prompts, images=images,
                                    return_tensors="pt", padding=True)
            inputs = {k: v.to(cap_model.device) if torch.is_tensor(v) else v
                      for k, v in inputs.items()}

            with torch.inference_mode():
                gen_ids = cap_model.generate(**inputs, max_new_tokens=100,
                                              do_sample=False, num_beams=1)

            input_len = inputs["input_ids"].shape[1]
            for j, (_, row) in enumerate(chunk.iterrows()):
                text = cap_processor.tokenizer.decode(
                    gen_ids[j, input_len:], skip_special_tokens=True).strip()
                all_captions[str(row["id"])] = text

            del inputs, gen_ids
            if (start // 8) % 20 == 0:
                torch.cuda.empty_cache()

        # Checkpoint after each split
        with open(caption_file, "w") as f:
            json.dump(all_captions, f)

    del cap_model, cap_processor
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\nDone! {len(all_captions)} captions saved.")

# Verify
expected = len(train_df) + len(val_df) + len(test_df)
print(f"Captions: {len(all_captions)} / {expected} expected")
sample_id = list(all_captions.keys())[0]
print(f"Sample ({sample_id}): {all_captions[sample_id][:150]}...")

Loaded 5165 cached captions.
Captions: 5165 / 5165 expected
Sample (train_07667): The image is a photograph of two frogs on a green surface. The frogs are positioned in such a way that their front legs are on top of each other, and ...


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6: Prompt Builders (3 variants for diversity)
# ══════════════════════════════════════════════════════════════════════════════

def _trunc(text, max_chars=MAX_CONTEXT_CHARS):
    if pd.isna(text): return ""
    text = str(text).strip()
    if len(text) <= max_chars: return text
    return text[:max_chars].rsplit(" ", 1)[0] + "..."


def prompt_caption_meta(row, captions, include_answer=False):
    """Starter format + caption + subject/grade metadata."""
    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    lecture = _trunc(row.get("lecture", ""))
    hint = _trunc(row.get("hint", ""))
    caption = captions.get(str(row["id"]), "")
    subject = str(row.get("subject", "")).strip() if pd.notna(row.get("subject")) else ""
    grade = str(row.get("grade", "")).strip() if pd.notna(row.get("grade")) else ""

    context_parts = []
    if caption: context_parts.append(f"Image description: {caption[:300]}")
    if lecture: context_parts.append(lecture)
    if hint: context_parts.append(hint)
    context_str = "\n".join(context_parts)

    prompt = "<image>\n"
    if subject or grade:
        meta = []
        if subject: meta.append(f"Subject: {subject}")
        if grade: meta.append(f"Grade: {grade}")
        prompt += " | ".join(meta) + "\n\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"

    return prompt


def prompt_caption_meta_solution(row, captions, include_answer=False):
    """Same but training answer includes solution."""
    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    lecture = _trunc(row.get("lecture", ""))
    hint = _trunc(row.get("hint", ""))
    caption = captions.get(str(row["id"]), "")
    subject = str(row.get("subject", "")).strip() if pd.notna(row.get("subject")) else ""
    grade = str(row.get("grade", "")).strip() if pd.notna(row.get("grade")) else ""

    context_parts = []
    if caption: context_parts.append(f"Image description: {caption[:300]}")
    if lecture: context_parts.append(lecture)
    if hint: context_parts.append(hint)
    context_str = "\n".join(context_parts)

    prompt = "<image>\n"
    if subject or grade:
        meta = []
        if subject: meta.append(f"Subject: {subject}")
        if grade: meta.append(f"Grade: {grade}")
        prompt += " | ".join(meta) + "\n\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        ans = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[ans]}"
        solution = _trunc(row.get("solution", ""), 200)
        if solution:
            prompt += f" Explanation: {solution}"

    return prompt


def prompt_no_caption(row, captions, include_answer=False):
    """Pure 0.859 team format — no caption, no metadata."""
    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    lecture = _trunc(row.get("lecture", ""))
    hint = _trunc(row.get("hint", ""))

    context_parts = []
    if lecture: context_parts.append(lecture)
    if hint: context_parts.append(hint)
    context_str = "\n".join(context_parts)

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"

    return prompt


print("3 prompt styles ready.")
row = combined_df.iloc[0]
print(f"\nCaption+meta prompt length: {len(prompt_caption_meta(row, all_captions, True))} chars")
print(f"No-caption prompt length: {len(prompt_no_caption(row, all_captions, True))} chars")

3 prompt styles ready.

Caption+meta prompt length: 1664 chars
No-caption prompt length: 1302 chars


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7: Dataset + Collator
# ══════════════════════════════════════════════════════════════════════════════

class TrainDataset(Dataset):
    def __init__(self, df, processor, data_dir, captions, prompt_fn):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.data_dir = data_dir
        self.captions = captions
        self.prompt_fn = prompt_fn

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(self.data_dir / row["image_path"]).convert("RGB")
    
        full_text = self.prompt_fn(row, self.captions, include_answer=True)
        prompt_text = self.prompt_fn(row, self.captions, include_answer=False)
    
        full_enc = self.processor(
            text=[full_text], images=[image],
            return_tensors="pt", truncation=False
        )
        prompt_enc = self.processor(
            text=[prompt_text], images=[image],
            return_tensors="pt", truncation=False
        )
    
        return {
            "input_ids": full_enc["input_ids"].squeeze(0),
            "attention_mask": full_enc["attention_mask"].squeeze(0),
            "pixel_values": full_enc["pixel_values"].squeeze(0),   # [num_tiles, C, H, W]
            "prompt_len": prompt_enc["input_ids"].shape[1],
            # no need to carry pixel_attention_mask — collator builds it
        }


def collate_fn(batch, pad_id):
    """Left-pad token sequences; pad tile dimension of pixel_values."""
    max_len = max(x["input_ids"].shape[0] for x in batch)
    max_tiles = max(x["pixel_values"].shape[0] for x in batch)

    ids, masks, labels, pixels, pixel_masks = [], [], [], [], []

    for x in batch:
        seq_len = x["input_ids"].shape[0]
        pad_len = max_len - seq_len

        # ── token padding (unchanged) ──
        padded_ids = torch.cat([
            torch.full((pad_len,), pad_id, dtype=x["input_ids"].dtype),
            x["input_ids"]
        ])
        padded_mask = torch.cat([
            torch.zeros(pad_len, dtype=x["attention_mask"].dtype),
            x["attention_mask"]
        ])
        lab = torch.full_like(padded_ids, -100)
        answer_start = pad_len + x["prompt_len"]
        lab[answer_start:] = padded_ids[answer_start:]

        # ── tile padding ──
        num_tiles, C, H, W = x["pixel_values"].shape
        tile_pad = max_tiles - num_tiles

        if tile_pad > 0:
            pv = torch.cat([
                x["pixel_values"],
                torch.zeros(tile_pad, C, H, W, dtype=x["pixel_values"].dtype)
            ], dim=0)
            # pixel_attention_mask: [num_tiles, H, W] → pad with zeros
            real_mask = torch.ones(num_tiles, H, W, dtype=torch.bool)
            pad_mask  = torch.zeros(tile_pad, H, W, dtype=torch.bool)
            pm = torch.cat([real_mask, pad_mask], dim=0)
        else:
            pv = x["pixel_values"]
            pm = torch.ones(num_tiles, H, W, dtype=torch.bool)

        ids.append(padded_ids)
        masks.append(padded_mask)
        labels.append(lab)
        pixels.append(pv)
        pixel_masks.append(pm)

    return {
        "input_ids": torch.stack(ids),
        "attention_mask": torch.stack(masks),
        "labels": torch.stack(labels),
        "pixel_values": torch.stack(pixels),                  # [B, max_tiles, C, H, W]
        "pixel_attention_mask": torch.stack(pixel_masks),    # [B, max_tiles, H, W]
    }

print("Dataset + Collator ready.")

Dataset + Collator ready.


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8: Model Loader (QLoRA + DoRA)
# ══════════════════════════════════════════════════════════════════════════════
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

def load_model(seed, lora_r=8, lora_alpha=16, use_dora=False,
               targets=None):
    if targets is None:
        targets = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        device_map="auto", low_cpu_mem_usage=True,
    )
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=lora_r, lora_alpha=lora_alpha, lora_dropout=0.05,
        target_modules=targets, bias="none",
        task_type=TaskType.CAUSAL_LM, use_dora=use_dora,
    )
    model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    dora_str = "DoRA" if use_dora else "LoRA"
    status = "OK" if trainable <= 5_000_000 else "OVER 5M LIMIT!"
    print(f"{dora_str} r={lora_r} a={lora_alpha} | {trainable:,} params ({trainable/1e6:.2f}M) | {status}")
    print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

    return model, processor

print("Model loader ready.")

Model loader ready.


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9: Training Function (flat LR, 3 epochs)
# ══════════════════════════════════════════════════════════════════════════════

def train_model(model, processor, train_df, captions, prompt_fn,
                exp_name, seed=42, epochs=3, lr=2e-4):
    print(f"\n{'='*60}")
    print(f"TRAINING: {exp_name}")
    print(f"{'='*60}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    pad_id = processor.tokenizer.pad_token_id
    dataset = TrainDataset(train_df, processor, DATA_DIR, captions, prompt_fn)
    loader = DataLoader(
        dataset, batch_size=TRAIN_BATCH, shuffle=True,
        collate_fn=lambda b: collate_fn(b, pad_id),
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    # FLAT LR — no scheduler (0.859 team finding)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.0,
    )

    total_batches = len(loader) * epochs
    print(f"Samples: {len(dataset)} | Batches/epoch: {len(loader)} | Total: {total_batches}")
    print(f"LR: {lr} FLAT | batch={TRAIN_BATCH} x accum={GRAD_ACCUM} = eff {TRAIN_BATCH*GRAD_ACCUM}")

    model.train()
    log = []
    start_time = time.time()

    for epoch in range(epochs):
        epoch_losses = []
        optimizer.zero_grad()

        pbar = tqdm(enumerate(loader), total=len(loader),
                    desc=f"Epoch {epoch+1}/{epochs}", leave=True)

        for batch_idx, batch in pbar:
            batch = {k: v.to(model.device) if torch.is_tensor(v) else v
                     for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM
            loss.backward()

            epoch_losses.append(outputs.loss.item())

            if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            avg_loss = np.mean(epoch_losses[-50:])
            pbar.set_postfix({
                "loss": f"{outputs.loss.item():.4f}",
                "avg": f"{avg_loss:.4f}",
                "VRAM": f"{torch.cuda.memory_allocated()/1e9:.1f}GB",
            })

            # Print every 100 steps (for HPC logs without tqdm)
            if (batch_idx + 1) % 100 == 0:
                elapsed_min = (time.time() - start_time) / 60
                log.append({"epoch": epoch+1, "batch": batch_idx+1,
                            "loss": outputs.loss.item(), "avg": avg_loss,
                            "elapsed_min": elapsed_min})
                print(f"  [{elapsed_min:.0f}min] Ep{epoch+1} Step {batch_idx+1}/{len(loader)}"
                      f" | loss={outputs.loss.item():.4f} avg={avg_loss:.4f}")

            # Checkpoint every 25%
            ckpt_interval = max(len(loader) // 4, 1)
            if (batch_idx + 1) % ckpt_interval == 0 and (batch_idx + 1) < len(loader):
                p = CHECKPOINT_DIR / f"{exp_name}_ep{epoch+1}_b{batch_idx+1}"
                model.save_pretrained(str(p))
                print(f"  Checkpoint: {p.name}")

        epoch_avg = np.mean(epoch_losses)
        print(f"\nEpoch {epoch+1} complete — Avg loss: {epoch_avg:.4f}")

        # End-of-epoch checkpoint
        ep_path = CHECKPOINT_DIR / f"{exp_name}_ep{epoch+1}_final"
        model.save_pretrained(str(ep_path))
        print(f"  Saved: {ep_path.name}")

    elapsed = time.time() - start_time
    final_path = CHECKPOINT_DIR / f"{exp_name}_final"
    model.save_pretrained(str(final_path))

    with open(CHECKPOINT_DIR / f"{exp_name}_log.json", "w") as f:
        json.dump(log, f, indent=2)

    print(f"\nDone: {final_path.name} | {elapsed/60:.1f} min total")

    del optimizer, loader, dataset
    gc.collect()
    torch.cuda.empty_cache()

    return model

print("Training function ready.")

Training function ready.


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10: Inference (fixed padding + multi-form tokens)
# ══════════════════════════════════════════════════════════════════════════════

def get_candidate_ids(tokenizer):
    cids = {}
    for letter in CHOICE_LETTERS:
        ids = set()
        for form in [letter, " " + letter]:
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc: ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids


def predict_batch(model, processor, df_batch, captions, prompt_fn, cand_ids):
    images = [Image.open(DATA_DIR / row["image_path"]).convert("RGB")
              for _, row in df_batch.iterrows()]
    prompts = [prompt_fn(row, captions, include_answer=False)
               for _, row in df_batch.iterrows()]

    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
              for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits

    # FIXED: correct last position per sample
    seq_lengths = inputs["attention_mask"].sum(dim=1)
    preds, scores_all = [], []

    for i, (_, row) in enumerate(df_batch.iterrows()):
        last_pos = int(seq_lengths[i].item()) - 1
        lp = torch.log_softmax(logits[i, last_pos, :], dim=-1)
        scores = [max(lp[tid].item() for tid in cand_ids[CHOICE_LETTERS[ci]])
                  for ci in range(len(row["choices"]))]
        preds.append(int(np.argmax(scores)))
        scores_all.append(scores)

    return preds, scores_all


def run_inference(model, processor, df, captions, prompt_fn, desc="Infer"):
    model.eval()
    cand_ids = get_candidate_ids(processor.tokenizer)
    all_preds, all_scores = [], []

    pbar = tqdm(range(0, len(df), INFER_BATCH), desc=desc, leave=True)
    for start in pbar:
        end = min(start + INFER_BATCH, len(df))
        preds, scores = predict_batch(model, processor, df.iloc[start:end],
                                       captions, prompt_fn, cand_ids)
        all_preds.extend(preds)
        all_scores.extend(scores)
        pbar.set_postfix({"done": len(all_preds)})
        torch.cuda.empty_cache()

    return all_preds, all_scores

print("Inference ready.")

Inference ready.


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 11: Ensemble Configs (5 diverse models)
# ══════════════════════════════════════════════════════════════════════════════

CONFIGS = [
    {
        "name": "A_cap_meta_dora_r8a16_s42",
        "seed": 42, "lora_r": 8, "lora_alpha": 16,
        "use_dora": True, "prompt_fn": prompt_caption_meta,
        "lr": 2e-4, "epochs": 3,
    },
    {
        "name": "B_cap_meta_dora_r8a16_s123",
        "seed": 123, "lora_r": 8, "lora_alpha": 16,
        "use_dora": True, "prompt_fn": prompt_caption_meta,
        "lr": 2e-4, "epochs": 3,
    },
    {
        "name": "C_cap_sol_dora_r8a16_s456",
        "seed": 456, "lora_r": 8, "lora_alpha": 16,
        "use_dora": True, "prompt_fn": prompt_caption_meta_solution,
        "lr": 2e-4, "epochs": 3,
    },
    {
        "name": "D_cap_meta_dora_r8a32_s789",
        "seed": 789, "lora_r": 8, "lora_alpha": 32,
        "use_dora": True, "prompt_fn": prompt_caption_meta,
        "lr": 2e-4, "epochs": 3,
    },
    {
        "name": "E_nocap_lora_r8a16_s42",
        "seed": 42, "lora_r": 8, "lora_alpha": 16,
        "use_dora": False, "prompt_fn": prompt_no_caption,
        "lr": 2e-4, "epochs": 3,
    },
]

print(f"{len(CONFIGS)} models:")
for c in CONFIGS:
    d = "DoRA" if c["use_dora"] else "LoRA"
    print(f"  {c['name']}: {d} r={c['lora_r']} a={c['lora_alpha']} lr={c['lr']}")

5 models:
  A_cap_meta_dora_r8a16_s42: DoRA r=8 a=16 lr=0.0002
  B_cap_meta_dora_r8a16_s123: DoRA r=8 a=16 lr=0.0002
  C_cap_sol_dora_r8a16_s456: DoRA r=8 a=16 lr=0.0002
  D_cap_meta_dora_r8a32_s789: DoRA r=8 a=32 lr=0.0002
  E_nocap_lora_r8a16_s42: LoRA r=8 a=16 lr=0.0002


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 12: MAIN LOOP — Train + Infer ALL Models
# ══════════════════════════════════════════════════════════════════════════════

all_results = []
overall_start = time.time()

for i, cfg in enumerate(CONFIGS):
    model_start = time.time()
    print(f"\n\n{'#'*60}")
    print(f"# MODEL {i+1}/{len(CONFIGS)}: {cfg['name']}")
    print(f"# Time elapsed: {(model_start - overall_start)/60:.0f} min")
    print(f"{'#'*60}")

    results_path = CHECKPOINT_DIR / f"{cfg['name']}_results.json"

    # CACHE: skip if done
    if results_path.exists():
        print(f"CACHED — loading {results_path.name}")
        with open(results_path) as f:
            all_results.append(json.load(f))
        continue

    # Load model
    model, processor = load_model(
        seed=cfg["seed"], lora_r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"], use_dora=cfg["use_dora"],
    )

    # Train on combined
    model = train_model(
        model, processor, combined_df, all_captions,
        prompt_fn=cfg["prompt_fn"], exp_name=cfg["name"],
        seed=cfg["seed"], epochs=cfg["epochs"], lr=cfg["lr"],
    )

    # Test inference
    print(f"\nTest inference...")
    test_preds, test_scores = run_inference(
        model, processor, test_df, all_captions,
        prompt_fn=cfg["prompt_fn"], desc=f"Test {cfg['name'][:20]}"
    )

    # Save
    result = {
        "name": cfg["name"],
        "test_preds": test_preds,
        "test_scores": test_scores,
        "train_time_min": (time.time() - model_start) / 60,
    }
    all_results.append(result)

    with open(results_path, "w") as f:
        json.dump(result, f)

    sub = pd.DataFrame({"id": test_df["id"], "answer": test_preds})
    sub.to_csv(SUBMISSION_DIR / f"sub_{cfg['name']}.csv", index=False)
    print(f"Saved sub_{cfg['name']}.csv")

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM freed. Model time: {(time.time()-model_start)/60:.1f} min")

total_time = (time.time() - overall_start) / 60
print(f"\n{'='*60}")
print(f"ALL {len(CONFIGS)} MODELS COMPLETE | Total: {total_time:.0f} min")
print(f"{'='*60}")



############################################################
# MODEL 1/5: A_cap_meta_dora_r8a16_s42
# Time elapsed: 0 min
############################################################
CACHED — loading A_cap_meta_dora_r8a16_s42_results.json


############################################################
# MODEL 2/5: B_cap_meta_dora_r8a16_s123
# Time elapsed: 0 min
############################################################
CACHED — loading B_cap_meta_dora_r8a16_s123_results.json


############################################################
# MODEL 3/5: C_cap_sol_dora_r8a16_s456
# Time elapsed: 0 min
############################################################
CACHED — loading C_cap_sol_dora_r8a16_s456_results.json


############################################################
# MODEL 4/5: D_cap_meta_dora_r8a32_s789
# Time elapsed: 0 min
############################################################
CACHED — loading D_cap_meta_dora_r8a32_s789_results.json


##############################

In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 13: Ensemble + Submissions
# ══════════════════════════════════════════════════════════════════════════════

test_nc = [len(row["choices"]) for _, row in test_df.iterrows()]
pred_lists = [r["test_preds"] for r in all_results]
score_lists = [r["test_scores"] for r in all_results]

# Majority vote
mv = [Counter([pl[i] for pl in pred_lists]).most_common(1)[0][0]
      for i in range(len(pred_lists[0]))]

# Score average
sa = []
for i in range(len(score_lists[0])):
    avg = [np.mean([sl[i][ci] for sl in score_lists if ci < len(sl[i])])
           for ci in range(test_nc[i])]
    sa.append(int(np.argmax(avg)))

# Save everything
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

all_subs = {"majority_vote": mv, "score_average": sa}
for r in all_results:
    all_subs[r["name"]] = r["test_preds"]

for name, preds in all_subs.items():
    sub = pd.DataFrame({"id": test_df["id"], "answer": preds})
    assert len(sub) == len(sample_sub), f"Length mismatch: {name}"
    sub.to_csv(SUBMISSION_DIR / f"sub_{name}.csv", index=False)

# Default = score average
pd.DataFrame({"id": test_df["id"], "answer": sa}).to_csv(
    SUBMISSION_DIR / "submission.csv", index=False)

print(f"{'='*60}")
print(f"SUBMISSIONS SAVED to {SUBMISSION_DIR}")
print(f"{'='*60}")
for name in all_subs:
    print(f"  sub_{name}.csv")
print(f"  submission.csv (= score_average)")
print(f"\nUpload ALL of these to Kaggle. Submit each one to find the best.")
print(f"\nPrediction distribution (score_average):")
print(pd.Series(sa).value_counts().sort_index())

SUBMISSIONS SAVED to /scratch/au2414/kaggle_final_competition/submissions_v2
  sub_majority_vote.csv
  sub_score_average.csv
  sub_A_cap_meta_dora_r8a16_s42.csv
  sub_B_cap_meta_dora_r8a16_s123.csv
  sub_C_cap_sol_dora_r8a16_s456.csv
  sub_D_cap_meta_dora_r8a32_s789.csv
  sub_E_nocap_lora_r8a16_s42.csv
  submission.csv (= score_average)

Upload ALL of these to Kaggle. Submit each one to find the best.

Prediction distribution (score_average):
0    363
1    352
2    217
3     75
4      1
Name: count, dtype: int64


In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 14: Summary
# ══════════════════════════════════════════════════════════════════════════════

summary = {
    "n_models": len(all_results),
    "models": [{"name": r["name"],
                "train_time_min": r.get("train_time_min", "cached")}
               for r in all_results],
    "training_data": f"train+val combined ({len(combined_df)} samples)",
    "base_recipe": "0.859 team: r=8, all 7 targets, flat LR=2e-4, 3 epochs",
    "our_additions": [
        "Self-captioning (SmolVLM image descriptions)",
        "DoRA (4 of 5 models)",
        "Subject/grade metadata in prompt",
        "Solution field (model C)",
        "Fixed padding inference",
        "5-model ensemble",
    ],
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    "submissions": list(all_subs.keys()),
}

with open(CHECKPOINT_DIR / "summary_v2.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f"\nDone! Upload submissions to Kaggle.")

{
  "n_models": 5,
  "models": [
    {
      "name": "A_cap_meta_dora_r8a16_s42",
      "train_time_min": 174.5402403195699
    },
    {
      "name": "B_cap_meta_dora_r8a16_s123",
      "train_time_min": 173.828116218249
    },
    {
      "name": "C_cap_sol_dora_r8a16_s456",
      "train_time_min": 177.06997725963592
    },
    {
      "name": "D_cap_meta_dora_r8a32_s789",
      "train_time_min": 174.64677548408508
    },
    {
      "name": "E_nocap_lora_r8a16_s42",
      "train_time_min": 103.91011769771576
    }
  ],
  "training_data": "train+val combined (4157 samples)",
  "base_recipe": "0.859 team: r=8, all 7 targets, flat LR=2e-4, 3 epochs",
  "our_additions": [
    "Self-captioning (SmolVLM image descriptions)",
    "DoRA (4 of 5 models)",
    "Subject/grade metadata in prompt",
    "Solution field (model C)",
    "Fixed padding inference",
    "5-model ensemble"
  ],
  "gpu": "N/A",
  "submissions": [
    "majority_vote",
    "score_average",
    "A_cap_meta_dora_r8a16_s42",

In [15]:
import json, numpy as np, pandas as pd
from collections import Counter

# Load all results
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_v2"
SUBMISSION_DIR = PROJECT_ROOT / "submissions_v2"

results = []
for name in ["A_cap_meta_dora_r8a16_s42", "B_cap_meta_dora_r8a16_s123",
             "D_cap_meta_dora_r8a32_s789", "E_nocap_lora_r8a16_s42"]:
    with open(CHECKPOINT_DIR / f"{name}_results.json") as f:
        results.append(json.load(f))

# Score average WITHOUT Model C
test_df = pd.read_csv(DATA_DIR / "test.csv")
test_df["choices"] = test_df["choices"].apply(json.loads)
test_nc = [len(row["choices"]) for _, row in test_df.iterrows()]

score_lists = [r["test_scores"] for r in results]
sa_no_c = []
for i in range(len(score_lists[0])):
    avg = [np.mean([sl[i][ci] for sl in score_lists if ci < len(sl[i])])
           for ci in range(test_nc[i])]
    sa_no_c.append(int(np.argmax(avg)))

sub = pd.DataFrame({"id": test_df["id"], "answer": sa_no_c})
sub.to_csv(SUBMISSION_DIR / "sub_ensemble_no_C.csv", index=False)

# Also try: only A + E (best individual models)
ae_scores = [results[0]["test_scores"], results[3]["test_scores"]]
sa_ae = []
for i in range(len(ae_scores[0])):
    avg = [np.mean([sl[i][ci] for sl in ae_scores if ci < len(sl[i])])
           for ci in range(test_nc[i])]
    sa_ae.append(int(np.argmax(avg)))

sub_ae = pd.DataFrame({"id": test_df["id"], "answer": sa_ae})
sub_ae.to_csv(SUBMISSION_DIR / "sub_ensemble_A_E_only.csv", index=False)

# Try A + B + E (drop C and D)
abe_scores = [results[0]["test_scores"], results[1]["test_scores"], results[3]["test_scores"]]
sa_abe = []
for i in range(len(abe_scores[0])):
    avg = [np.mean([sl[i][ci] for sl in abe_scores if ci < len(sl[i])])
           for ci in range(test_nc[i])]
    sa_abe.append(int(np.argmax(avg)))

sub_abe = pd.DataFrame({"id": test_df["id"], "answer": sa_abe})
sub_abe.to_csv(SUBMISSION_DIR / "sub_ensemble_A_B_E.csv", index=False)

print("Generated 3 new ensemble submissions:")
print("  sub_ensemble_no_C.csv (A+B+D+E)")
print("  sub_ensemble_A_E_only.csv (A+E)")
print("  sub_ensemble_A_B_E.csv (A+B+E)")

Generated 3 new ensemble submissions:
  sub_ensemble_no_C.csv (A+B+D+E)
  sub_ensemble_A_E_only.csv (A+E)
  sub_ensemble_A_B_E.csv (A+B+E)
